# Anime Action Scene — 2D Baseline Inference
**Moore-AnimateAnyone + Anything V5**

순서대로 셀 실행하면 됩니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **A100 GPU** 선택 후 시작

In [ ]:
# ── 0. GPU 확인 ───────────────────────────────────────────────────────────────
import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── 1. 레포 클론 ───────────────────────────────────────────────────────────────
import os

if not os.path.exists('anime-action-scene'):
    !git clone https://github.com/goldkangsan/anime-action-scene.git
else:
    print('이미 클론됨. 최신 버전으로 업데이트...')
    !cd anime-action-scene && git pull

%cd anime-action-scene
!ls

In [ ]:
# ── 2. 의존성 설치 ─────────────────────────────────────────────────────────────
# 버전 충돌 방지를 위해 순서 중요
!pip install -q --upgrade pip

# 핵심 버전 고정
!pip install -q \
    diffusers==0.24.0 \
    transformers==4.30.2 \
    accelerate \
    huggingface_hub

# 나머지
!pip install -q \
    omegaconf \
    einops \
    controlnet-aux==0.0.7 \
    onnxruntime-gpu \
    imageio \
    imageio[ffmpeg] \
    av \
    gradio

print('\n✅ 설치 완료')

In [ ]:
# ── 3. import 테스트 ───────────────────────────────────────────────────────────
# 설치 후 런타임 재시작 없이 바로 확인
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from src.dwpose import DWposeDetector
from src.models.pose_guider import PoseGuider
from src.models.unet_2d_condition import UNet2DConditionModel
from src.models.unet_3d import UNet3DConditionModel
from src.pipelines.pipeline_pose2vid_long import Pose2VideoPipeline
from src.utils.util import get_fps, read_frames, save_videos_grid

print('✅ 모든 import 성공!')

In [ ]:
# ── 4. 가중치 다운로드 (~4GB, 처음 한 번만) ───────────────────────────────────
# 이미 있으면 자동 스킵됨
!python tools/download_weights.py

# Anything V5 추가 (애니 스타일용)
!python tools/download_weights.py --anime

In [ ]:
# ── 5. 입력 파일 업로드 ────────────────────────────────────────────────────────
import shutil
from google.colab import files
from pathlib import Path

Path('inputs').mkdir(exist_ok=True)

print('📁 캐릭터 이미지를 업로드하세요 (jpg/png)')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'inputs/ref.png')
    print(f'  → inputs/ref.png 저장 완료')

In [ ]:
print('🎬 액션 영상을 업로드하세요 (mp4)')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'inputs/driving.mp4')
    print(f'  → inputs/driving.mp4 저장 완료')

In [ ]:
# ── 6. 업로드 확인 ─────────────────────────────────────────────────────────────
from PIL import Image
import cv2
from IPython.display import display

# 참조 이미지 확인
ref = Image.open('inputs/ref.png')
print(f'참조 이미지: {ref.size}')
display(ref.resize((200, int(200 * ref.height / ref.width))))

# 드라이빙 영상 확인
cap = cv2.VideoCapture('inputs/driving.mp4')
fps  = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'드라이빙 영상: {total}프레임 @ {fps:.1f}fps')

In [ ]:
# ── 7. DWPose 추출 (영상 → 포즈 비디오) ───────────────────────────────────────
!python tools/vid2pose.py \
    --video_path inputs/driving.mp4 \
    --output_path outputs/pose/driving_kps.mp4 \
    --device cuda \
    --max_frames 32

In [ ]:
# ── 8-A. Inference — SD 1.5 (비교용 baseline) ─────────────────────────────────
!python scripts/pose2vid.py \
    --config ./configs/prompts/animation.yaml \
    -W 384 -H 512 -L 32 \
    --steps 20 \
    --cfg 3.5 \
    --seed 42 \
    --device cuda

In [ ]:
# ── 8-B. Inference — Anything V5 (애니 특화) ──────────────────────────────────
!python scripts/pose2vid.py \
    --config ./configs/prompts/animation_anime.yaml \
    -W 384 -H 512 -L 32 \
    --steps 20 \
    --cfg 3.5 \
    --seed 42 \
    --device cuda

In [ ]:
# ── 9. 결과 확인 ───────────────────────────────────────────────────────────────
import glob
from IPython.display import Video, display

results = sorted(glob.glob('output/**/*.mp4', recursive=True))
print(f'생성된 영상 {len(results)}개:')
for r in results:
    print(f'  {r}')

if results:
    print('\n▶ 최신 결과 미리보기:')
    display(Video(results[-1], width=512, embed=True))

In [ ]:
# ── 10. 결과 다운로드 ──────────────────────────────────────────────────────────
!zip -r results.zip output/

from google.colab import files
files.download('results.zip')
print('✅ results.zip 다운로드 시작')